# Lab 3 v2 — Exercise 1: WordPiece Tokenizer & Hugging Face Sentiment Pipeline

**Mục tiêu:** Hiểu trực quan toàn bộ luồng **`Raw Text → WordPiece Tokenizer (tokens, input_ids, attention_mask, decode) → Pretrained DistilBERT (SST-2) → Sentiment Label & Score`** bằng `transformers.pipeline`.

- **Model:** `distilbert-base-uncased-finetuned-sst-2-english`
- **CLI tương ứng:** `python labs/lab3_v2/run_ex1.py`
- **Tài liệu liên quan:** [`REPORT.md`](REPORT.md) · [`docs/models/distilbert.md`](docs/models/distilbert.md) · [`docs/experiment_report.md`](docs/experiment_report.md)

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
from IPython.display import display

REPO_ROOT = Path.cwd().resolve()
if not (REPO_ROOT / 'labs' / 'lab3_v2').exists():
    REPO_ROOT = REPO_ROOT.parents[1]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import labs.lab3_v2.config as cfg

RESULTS_PATH = cfg.EX1_DIR / 'ex1_results.json'
RUN_LIVE_IF_MISSING = False  # Đổi thành True nếu muốn chạy trực tiếp từ notebook thay vì CLI

if RESULTS_PATH.exists():
    ex1_data = json.loads(RESULTS_PATH.read_text(encoding='utf-8'))
    print(f'Loaded verified Ex1 artifact: {RESULTS_PATH}')
elif RUN_LIVE_IF_MISSING:
    from src.vibe_scripts.lab3_v2.ex1_sentiment import run_ex1_demo
    ex1_data = run_ex1_demo(model_name=cfg.EX1_MODEL_NAME, output_dir=cfg.EX1_DIR)
else:
    ex1_data = None
    print('Chưa có ex1_results.json — Hãy chạy Step 4 CLI: python labs/lab3_v2/run_ex1.py')

Loaded verified Ex1 artifact: D:\notworLe\Desktop\gitvanhub\Deeplearning-UTH\experiments\lab3_v2\ex1\ex1_results.json


## Bước 1 — Giải Phẫu WordPiece Tokenizer (`tokens`, `input_ids`, `attention_mask`, `decode`)

Tokenizer tách câu tiếng Anh thành các mảnh **WordPiece** (các mảnh nối tiếp mang tiền tố `##`), chèn `[CLS]` (`101`) ở đầu câu và `[SEP]` (`102`) ở cuối câu, ánh xạ sang số nguyên `input_ids`, và sinh `attention_mask` (`1` cho token thật, `0` cho `[PAD]`).

In [2]:
if ex1_data is not None:
    tok = ex1_data['tokenizer_demo']
    print('Original text        :', tok['original_text'])
    print('Num tokens           :', tok['num_tokens'])
    print('Subword pieces (##)  :', tok['subword_pieces'])
    print('Special tokens       :', tok['special_tokens_map'])
    print('Decoded (with spec)  :', tok['decoded_with_special'])
    print('Decoded (clean)      :', tok['decoded_clean'])
    tok_df = pd.DataFrame({
        'pos': list(range(tok['num_tokens'])),
        'token': tok['tokens'],
        'input_id': tok['input_ids'],
        'attention_mask': tok['attention_mask'],
    })
    display(tok_df)
else:
    print('Bỏ qua — chờ chạy `python labs/lab3_v2/run_ex1.py`.')

Original text        : Hugging Face transformers make fine-tuning unbelievable and surprisingly enjoyable!
Num tokens           : 14
Subword pieces (##)  : []
Special tokens       : {'cls_token': '[CLS]', 'cls_token_id': 101, 'sep_token': '[SEP]', 'sep_token_id': 102, 'pad_token': '[PAD]', 'pad_token_id': 0}
Decoded (with spec)  : [CLS] hugging face transformers make fine - tuning unbelievable and surprisingly enjoyable! [SEP]
Decoded (clean)      : hugging face transformers make fine - tuning unbelievable and surprisingly enjoyable!


,pos,token,input_id,attention_mask
0,0,[CLS],101,1
1,1,hugging,17662,1
2,2,face,2227,1
3,3,transformers,19081,1
4,4,make,2191,1
5,5,fine,2986,1
6,6,-,1011,1
7,7,tuning,17372,1
8,8,unbelievable,23653,1
9,9,and,1998,1


## Bước 2 — Minh Họa Batch Padding (`padding=True` → Tensor `(B, L)`)

Khi gom nhiều câu khác độ dài vào cùng 1 batch cho GPU, câu ngắn được thêm `[PAD]` (`id = 0`) và đánh dấu `attention_mask = 0` để Self-Attention bỏ qua hoàn toàn các vị trí đệm.

In [3]:
if ex1_data is not None:
    bp = ex1_data['batch_padding_demo']
    print(f'Tensor input_ids shape={bp["input_ids_shape"]} | attention_mask shape={bp["attention_mask_shape"]} | dtype={bp["dtype"]}')
    pad_df = pd.DataFrame([
        {
            'text': r['text'],
            'real_tokens': r['real_tokens'],
            'pad_tokens': r['pad_tokens'],
            'tokens': ' '.join(r['tokens']),
            'attention_mask': str(r['attention_mask']),
        }
        for r in bp['rows']
    ])
    display(pad_df)
else:
    print('Bỏ qua — chờ chạy `python labs/lab3_v2/run_ex1.py`.')

Tensor input_ids shape=[2, 11] | attention_mask shape=[2, 11] | dtype=torch.int64


,text,real_tokens,pad_tokens,tokens,attention_mask
0,Great movie!,5,6,[CLS] great movie ! [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD],"[1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0]"
1,This movie was surprisingly thoughtful and well directed.,11,0,[CLS] this movie was surprisingly thoughtful and well directed . [SEP],"[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]"


## Bước 3 — Phân Tích Cảm Xúc Trên 8 Câu Mẫu (Tích Cực, Tiêu Cực, Phủ Định & Mỉa Mai)

Quan sát nơi mô hình pretrained SST-2 hoạt động xuất sắc (câu rõ ràng, tương phản/phủ định) và nơi mô hình có thể bị đánh lừa (câu mỉa mai chứa từ tích cực bề mặt như *"Oh great"*, *"highlight of my vacation"*).

In [4]:
if ex1_data is not None:
    print(f'Model: {ex1_data["model_name"]} | Device: {ex1_data["device"]} | Matched: {ex1_data["matched_expected"]}/{ex1_data["num_samples"]}')
    pred_df = pd.DataFrame(ex1_data['predictions'])
    display(pred_df)
else:
    print('Bỏ qua — chờ chạy `python labs/lab3_v2/run_ex1.py`.')

Model: distilbert-base-uncased-finetuned-sst-2-english | Device: cuda | Matched: 5/8


,category,text,expected,predicted_label,score,matches_expected
0,positive_clear,I absolutely loved this movie! The acting was brilliant and the story kept me hooked from start to finish.,POSITIVE,POSITIVE,0.9999,True
1,positive_clear,This library makes working with pretrained transformer models surprisingly intuitive and fast.,POSITIVE,POSITIVE,0.9995,True
2,negative_clear,What a complete waste of two hours. The plot made zero sense and the dialogue was awful.,NEGATIVE,NEGATIVE,0.9998,True
3,negative_clear,"The service was painfully slow, the staff were rude, and the food arrived completely cold.",NEGATIVE,NEGATIVE,0.9997,True
4,negation_contrast,"I expected this film to be terrible, yet it turned out not bad at all and genuinely entertaining.",POSITIVE,POSITIVE,0.9992,True
5,mixed_contrast,"The cinematography is breathtaking, although the pacing drags terribly and the ending ruins everything.",NEGATIVE,POSITIVE,0.9968,False
6,sarcasm_hard,"Oh great, another three-hour sequel with absolutely nothing new to say. Just what I wanted.",NEGATIVE,POSITIVE,0.6246,False
7,sarcasm_hard,"Yeah, waiting four hours at a delayed airport gate was definitely the highlight of my vacation.",NEGATIVE,POSITIVE,0.9947,False
